## Synthetic Campaign Data

### CSV

In [1]:
import pandas as pd
from faker import Faker
import random

fake = Faker()

# CSV File
csv_folder = "data/csv"
campaign_file_name = "campaign_data.csv"
campaign_file_full_path = f"{csv_folder}/{campaign_file_name}"
print("campaign_file_full_path:", campaign_file_full_path)

def generate_synthetic_campaigns():

    def generate_campaign_data(num_campaigns=1000):
        campaign_data = []

        campaign_topics = [
            "Seasonal Discount", "New Product Launch", "School Supplies Promo",
            "Holiday Sales Event", "Beta Testing Launch", "Major Shopping Event",
            "Clearance Sale", "Fitness Gear Promo", "Sustainability Product Launch",
            "Holiday Follow Up", "Brand Awareness", "Loyalty Program"
        ]

        customer_segments = [
            "Women 25-34", "Tech Enthusiasts 18-45", "Parents 30-50",
            "General Audience", "Young Adults 20-30", "Shopaholics 25-44",
            "Seniors 55+", "Fitness Enthusiasts 18-40", "Environment Advocates",
            "Previous Customers", "Young Professionals 25-40", "Frequent Buyers 35-50"
        ]

        for i in range(1, num_campaigns + 1):
            campaign_id = f"{i}"
            # Campaign date randomized in last 18 months up to today (July 24, 2025)
            campaign_date = fake.date_between(start_date='-18M', end_date='today')
            campaign_topic = random.choice(campaign_topics)
            customer_segment = random.choice(customer_segments)

            # Audience size between 50,000 and 200,000
            audience_size = random.randint(50000, 200000)

            # Target description — same as customer segment for simplicity
            target_desc = customer_segment  

            # control_group_size about 5-10% of audience
            control_group_size = int(audience_size * random.uniform(0.05, 0.10))

            # Sent is audience size minus control group
            sent = audience_size - control_group_size

            # Open rate between 15% - 30%
            open_rate = random.uniform(15, 30) / 100

            # Click rate between 3% - 12%
            click_rate = random.uniform(3, 12) / 100

            # Conversions between 0.5% - 7%
            conversion_rate = random.uniform(0.5, 7) / 100

            # Opens = sent * open_rate (rounded)
            opens = int(sent * open_rate)

            # Clicks = sent * click_rate (rounded)
            clicks = int(sent * click_rate)

            # Conversions = sent * conversion_rate (rounded)
            conversions = int(sent * conversion_rate)

            # Open to click rate = clicks / opens
            open_to_click_rate = (clicks / opens) if opens > 0 else 0

            campaign_data.append({
                "campaign_id": campaign_id,
                "campaign_date": campaign_date,
                "campaign_topic": campaign_topic,
                "customer_segment": customer_segment,
                "audience_size": audience_size,
                "target_description": target_desc,
                "control_group_size": control_group_size,
                "sent": sent,
                "opens": opens,
                "clicks": clicks,
                "conversions": conversions,
                "open_rate": round(open_rate * 100, 2),
                "click_rate": round(click_rate * 100, 2),
                "open_to_click_rate": round(open_to_click_rate * 100, 2),
                "conversion_rate": round(conversion_rate * 100, 2)
            })

        df = pd.DataFrame(campaign_data)
        return df
    
    # Generate synthetic campaigns
    df_campaigns = generate_campaign_data(1000)

    # Save to CSV
    df_campaigns.to_csv(campaign_file_full_path, index=False)



campaign_file_full_path: data/csv/campaign_data.csv


In [2]:
generate_synthetic_campaigns()


## Campaign Summary Reports

### DOCX

In [3]:
import pandas as pd
from docx import Document
from docx.shared import Pt
from datetime import datetime

def generate_campaign_summary_report_as_docx():
    
    def generate_docx_campaign_summary_report(df, campaign_index=0, output_path="campaign_summary_report.docx"):
        doc = Document()

        # Extract the campaign data row
        campaign = df.iloc[campaign_index]
        
        # Create title
        doc.add_heading(f'Marketing Campaign Summary Report: Campaign {campaign["campaign_id"]}', level=1)
        
        # Add metadata
        doc.add_paragraph(f'Report Date: {datetime.today().strftime("%Y-%m-%d")}')
        doc.add_paragraph(f'Campaign Date: {campaign["campaign_date"]}')
        doc.add_paragraph(f'Customer Segment: {campaign["customer_segment"]}')
        doc.add_paragraph(f'Campaign Topic: {campaign["campaign_topic"]}')
        
        doc.add_paragraph()  # blank line
        
        # Executive Summary
        doc.add_heading('Executive Summary', level=2)
        exec_summary = (
            f"The campaign {campaign['campaign_id']} targeted {campaign['customer_segment']} with the objective to "
            f"promote {campaign['campaign_topic'].lower()}. Out of an audience size of {campaign['audience_size']}, "
            f"emails were sent to {campaign['sent']:,} contacts, with a control group of {campaign['control_group_size']:,}. "
            f"The campaign achieved an open rate of {campaign['open_rate']:.2f}%, click rate of {campaign['click_rate']:.2f}%, "
            f"and a conversion rate of {campaign['conversion_rate']:.2f}%, signifying strong engagement and effectiveness."
        )
        doc.add_paragraph(exec_summary)
        
        # Campaign Overview
        doc.add_heading('Campaign Overview', level=2)
        overview_table = doc.add_table(rows=6, cols=2)
        overview_table.style = 'LightShading-Accent1'
        overview_data = {
            "Campaign Id": campaign["campaign_id"],
            "Campaign Date": campaign["campaign_date"],
            "Campaign Topic": campaign["campaign_topic"],
            "Customer Segment": campaign["customer_segment"],
            "Audience Size": f"{campaign['audience_size']:,}",
            "control_group_size": f"{campaign['control_group_size']:,}",
        }
        for i, (key, value) in enumerate(overview_data.items()):
            overview_table.cell(i, 0).text = key
            overview_table.cell(i, 1).text = str(value)
        
        doc.add_paragraph()
        
        # Key Metrics
        doc.add_heading('Key Metrics', level=2)
        metrics_table = doc.add_table(rows=6, cols=3)
        metrics_table.style = 'LightShading-Accent2'
        # Table header
        hdr_cells = metrics_table.rows[0].cells
        hdr_cells[0].text = 'Metric'
        hdr_cells[1].text = 'Count'
        hdr_cells[2].text = 'Rate (%)'
        
        metric_data = [
            ("Emails Sent", f"{campaign['sent']:,}", "-"),
            ("Opens", f"{campaign['opens']:,}", f"{campaign['open_rate']:.2f}"),
            ("Clicks", f"{campaign['clicks']:,}", f"{campaign['click_rate']:.2f}"),
            ("Conversions", f"{campaign['conversions']:,}", f"{campaign['conversion_rate']:.2f}"),
            ("Open to Click Rate", "-", f"{campaign['open_to_click_rate']:.2f}"),
        ]
        
        for i, (metric, count, rate) in enumerate(metric_data, 1):
            row_cells = metrics_table.rows[i].cells
            row_cells[0].text = metric
            row_cells[1].text = count
            row_cells[2].text = rate
        
        doc.add_paragraph()
        
        # Performance Insights
        doc.add_heading('Performance Insights', level=2)
        insights = (
            f"- The open rate of {campaign['open_rate']:.2f}% indicates strong interest among the target audience.\n"
            f"- An open to click rate of {campaign['open_to_click_rate']:.2f}% suggests effective email content and calls-to-action.\n"
            f"- The conversion rate of {campaign['conversion_rate']:.2f}% reflects well on the campaign’s ability to drive results."
        )
        doc.add_paragraph(insights)
        
        # Recommendations
        doc.add_heading('Recommendations', level=2)
        recommendations = (
            "- Test subject lines and sending times to increase open rates.\n"
            "- Segment audience further for targeted offers.\n"
            "- Optimize landing pages to boost conversion rates."
        )
        doc.add_paragraph(recommendations)
        
        # Save the document
        doc.save(output_path)
        print(f"Summary report generated and saved as '{output_path}'.")

    # Load the CSV file you generated from the previous code
    df_campaigns = pd.read_csv(campaign_file_full_path, parse_dates=["campaign_date"])

    # Folder
    docs_folder = "data/doc"

    # Generate reports
    for i in range(0, 2):
        campaign = df_campaigns.iloc[i]
        campaign_id = campaign['campaign_id']
        doc_file_name = f"campaign_{campaign_id}_summary_report.docx"
        doc_file_full_path = f"{docs_folder}/{doc_file_name}"
        generate_docx_campaign_summary_report(df_campaigns, campaign_index=i, output_path=doc_file_full_path)



In [4]:
generate_campaign_summary_report_as_docx()

Summary report generated and saved as 'data/doc/campaign_1_summary_report.docx'.
Summary report generated and saved as 'data/doc/campaign_2_summary_report.docx'.


/Users/lolo/shared/projects/turing/projects/project-02/.venv/lib/python3.12/site-packages/docx/styles/styles.py:125: UserWarning: style lookup by style_id is deprecated. Use style name as key instead.
  return self._get_style_id_from_style(self[style_name], style_type)


### PDF

In [5]:
import pandas as pd
from reportlab.lib.pagesizes import LETTER
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from datetime import datetime

def generate_campaign_summary_report_as_pdf():

    def generate_campaign_pdf_report(df, campaign_index=0, output_path="campaign_1_summary_report.pdf"):
        campaign = df.iloc[campaign_index]

        doc = SimpleDocTemplate(output_path, pagesize=LETTER,
                                rightMargin=72, leftMargin=72,
                                topMargin=72, bottomMargin=18)

        styles = getSampleStyleSheet()
        styles.add(ParagraphStyle(name='CenterTitle', alignment=TA_CENTER, fontSize=16, spaceAfter=20))
        
        elements = []

        # Title
        elements.append(Paragraph(f"Marketing Campaign Summary Report: Campaign {campaign['campaign_id']}", styles['CenterTitle']))

        # Meta info
        meta_info = [
            f"Report Date: {datetime.today().strftime('%Y-%m-%d')}",
            f"Campaign Date: {campaign['campaign_date'].strftime('%Y-%m-%d') if not pd.isnull(campaign['campaign_date']) else 'N/A'}",
            f"Customer Segment: {campaign['customer_segment']}",
            f"Campaign Topic: {campaign['campaign_topic']}"
        ]
        for item in meta_info:
            elements.append(Paragraph(item, styles['BodyText']))

        elements.append(Spacer(1, 12))

        # Executive Summary
        elements.append(Paragraph("Executive Summary", styles['Heading2']))
        exec_summary = (
            f"The Campaign <b>{campaign['campaign_id']}</b> targeted <b>{campaign['customer_segment']}</b> with the objective to "
            f"promote <b>{campaign['campaign_topic'].lower()}</b>. Out of an audience size of <b>{campaign['audience_size']:,}</b>, "
            f"emails were sent to <b>{campaign['sent']:,}</b> contacts, with a control group of <b>{campaign['control_group_size']:,}</b>. "
            f"The campaign achieved an open rate of <b>{campaign['open_rate']:.2f}%</b>, click rate of <b>{campaign['click_rate']:.2f}%</b>, "
            f"and a conversion rate of <b>{campaign['conversion_rate']:.2f}%</b>, signifying strong engagement and effectiveness."
        )
        elements.append(Paragraph(exec_summary, styles['BodyText']))

        elements.append(Spacer(1, 12))

        # Campaign Overview Table
        elements.append(Paragraph("Campaign Overview", styles['Heading2']))

        overview_data = [
            ['Campaign Id', campaign["campaign_id"]],
            ['Campaign Date', campaign['campaign_date'].strftime('%Y-%m-%d') if not pd.isnull(campaign['campaign_date']) else 'N/A'],
            ['Campaign Topic', campaign['campaign_topic']],
            ['Customer Segment', campaign['customer_segment']],
            ['Audience Size', f"{campaign['audience_size']:,}"],
            ['Control Group Size', f"{campaign['control_group_size']:,}"],
        ]

        overview_table = Table(overview_data, hAlign='LEFT', colWidths=[150, 300])
        overview_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
            ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
            ('FONTSIZE', (0, 0), (-1, -1), 10),
            ('BOX', (0,0), (-1,-1), 1, colors.black),
            ('INNERGRID', (0,0), (-1,-1), 0.5, colors.grey)
        ]))
        elements.append(overview_table)
        elements.append(Spacer(1, 20))

        # Key Metrics Table
        elements.append(Paragraph("Key Metrics", styles['Heading2']))

        metrics_data = [
            ['Metric', 'Count', 'Rate (%)'],
            ['Emails Sent', f"{campaign['sent']:,}", '-'],
            ['Opens', f"{campaign['opens']:,}", f"{campaign['open_rate']:.2f}"],
            ['Clicks', f"{campaign['clicks']:,}", f"{campaign['click_rate']:.2f}"],
            ['Conversions', f"{campaign['conversions']:,}", f"{campaign['conversion_rate']:.2f}"],
            ['Open to Click Rate', '-', f"{campaign['open_to_click_rate']:.2f}"]
        ]

        metrics_table = Table(metrics_data, hAlign='LEFT', colWidths=[150, 100, 100])
        metrics_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.lightblue),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('ALIGN', (1, 1), (-1, -1), 'RIGHT'),
            ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
            ('FONTSIZE', (0, 0), (-1, -1), 10),
            ('BOX', (0,0), (-1,-1), 1, colors.black),
            ('INNERGRID', (0,0), (-1,-1), 0.5, colors.grey)
        ]))

        elements.append(metrics_table)
        elements.append(Spacer(1, 20))

        # Performance Insights
        elements.append(Paragraph("Performance Insights", styles['Heading2']))
        insights = (
            f"- The open rate of <b>{campaign['open_rate']:.2f}%</b> indicates strong interest among the target audience.<br/>"
            f"- An open to click rate of <b>{campaign['open_to_click_rate']:.2f}%</b> suggests effective email content and calls-to-action.<br/>"
            f"- The conversion rate of <b>{campaign['conversion_rate']:.2f}%</b> reflects well on the campaign’s ability to drive results."
        )
        elements.append(Paragraph(insights, styles['BodyText']))

        # Recommendations
        elements.append(Paragraph("Recommendations", styles['Heading2']))
        recommendations = (
            "- Test subject lines and sending times to increase open rates.<br/>"
            "- Segment audience further for targeted offers.<br/>"
            "- Optimize landing pages to boost conversion rates."
        )
        elements.append(Paragraph(recommendations, styles['BodyText']))

        doc.build(elements)
        print(f"PDF report generated and saved as '{output_path}'.")

    # -- Process
    # Load CSV generated by previous step
    df_campaigns = pd.read_csv(campaign_file_full_path, parse_dates=["campaign_date"])

    # Folder
    pdfs = "data/pdf"
    
    # Generate reports
    for i in range(100, 102):
        campaign = df_campaigns.iloc[i]
        campaign_id = campaign['campaign_id']
        pdf_file_name = f"campaign_{campaign_id}_summary_report.pdf"
        pdf_file_full_path = f"{pdfs}/{pdf_file_name}"
        generate_campaign_pdf_report(df_campaigns, campaign_index=i, output_path=pdf_file_full_path)



In [6]:
generate_campaign_summary_report_as_pdf()

PDF report generated and saved as 'data/pdf/campaign_101_summary_report.pdf'.
PDF report generated and saved as 'data/pdf/campaign_102_summary_report.pdf'.


### HTML

In [7]:
import pandas as pd
from datetime import datetime


def generate_campaign_summary_report_as_html():
    def generate_campaign_summary_html(df, campaign_index=0, output_path="campaign_1_summary_report.html"):
        campaign = df.iloc[campaign_index]
        
        # Format campaign date nicely
        campaign_date = campaign["campaign_date"]
        if pd.isnull(campaign_date):
            campaign_date_str = "N/A"
        elif isinstance(campaign_date, pd.Timestamp):
            campaign_date_str = campaign_date.strftime("%Y-%m-%d")
        else:
            campaign_date_str = str(campaign_date)
        
        html_content = f"""
        <!DOCTYPE html>
        <html lang="en">
        <head>
            <meta charset="UTF-8" />
            <meta name="viewport" content="width=device-width, initial-scale=1" />
            <title>Campaign Summary Report - Campaign {campaign['campaign_id']}</title>
            <style>
                body {{
                    font-family: Arial, sans-serif;
                    margin: 40px;
                    background-color: #f9f9f9;
                    color: #333;
                }}
                h1, h2 {{
                    color: #004080;
                }}
                table {{
                    width: 100%;
                    border-collapse: collapse;
                    margin-bottom: 20px;
                }}
                th, td {{
                    border: 1px solid #ccc;
                    padding: 8px 12px;
                    text-align: left;
                }}
                th {{
                    background-color: #007acc;
                    color: white;
                }}
                tr:nth-child(even) {{
                    background-color: #e6f0fa;
                }}
                .section {{
                    margin-bottom: 40px;
                }}
                .recommendations ul {{
                    list-style-type: disc;
                    margin-left: 20px;
                }}
            </style>
        </head>
        <body>
            <h1>Marketing Campaign Summary Report: Campaign {campaign['campaign_id']}</h1>
            <p><strong>Report Date:</strong> {datetime.today().strftime("%Y-%m-%d")}</p>
            <p><strong>Campaign Date:</strong> {campaign_date_str}</p>
            <p><strong>Customer Segment:</strong> {campaign['customer_segment']}</p>
            <p><strong>Campaign Topic:</strong> {campaign['campaign_topic']}</p>

            <div class="section">
                <h2>Executive Summary</h2>
                <p>
                    The Campaign <strong>{campaign['campaign_id']}</strong> campaign targeted <strong>{campaign['customer_segment']}</strong> 
                    with the objective to promote <strong>{campaign['campaign_topic'].lower()}</strong>. Out of an audience size 
                    of <strong>{campaign['audience_size']:,}</strong>, emails were sent to <strong>{campaign['sent']:,}</strong> contacts, 
                    with a control group of <strong>{campaign['control_group_size']:,}</strong>. The campaign achieved an open rate 
                    of <strong>{campaign['open_rate']:.2f}%</strong>, click rate of <strong>{campaign['click_rate']:.2f}%</strong>, 
                    and a conversion rate of <strong>{campaign['conversion_rate']:.2f}%</strong>, signifying strong engagement and effectiveness.
                </p>
            </div>

            <div class="section">
                <h2>Campaign Overview</h2>
                <table>
                    <tr><th>Metric</th><th>Value</th></tr>
                    <tr><td>Campaign Name</td><td>{campaign['campaign_id']}</td></tr>
                    <tr><td>Campaign Date</td><td>{campaign_date_str}</td></tr>
                    <tr><td>Campaign Topic</td><td>{campaign['campaign_topic']}</td></tr>
                    <tr><td>Customer Segment</td><td>{campaign['customer_segment']}</td></tr>
                    <tr><td>Audience Size</td><td>{campaign['audience_size']:,}</td></tr>
                    <tr><td>Control Group Size</td><td>{campaign['control_group_size']:,}</td></tr>
                </table>
            </div>

            <div class="section">
                <h2>Key Metrics</h2>
                <table>
                    <tr>
                        <th>Metric</th>
                        <th>Count</th>
                        <th>Rate (%)</th>
                    </tr>
                    <tr>
                        <td>Emails Sent</td>
                        <td>{campaign['sent']:,}</td>
                        <td>-</td>
                    </tr>
                    <tr>
                        <td>Opens</td>
                        <td>{campaign['opens']:,}</td>
                        <td>{campaign['open_rate']:.2f}</td>
                    </tr>
                    <tr>
                        <td>Clicks</td>
                        <td>{campaign['clicks']:,}</td>
                        <td>{campaign['click_rate']:.2f}</td>
                    </tr>
                    <tr>
                        <td>Conversions</td>
                        <td>{campaign['conversions']:,}</td>
                        <td>{campaign['conversion_rate']:.2f}</td>
                    </tr>
                    <tr>
                        <td>Open to Click Rate</td>
                        <td>-</td>
                        <td>{campaign['open_to_click_rate']:.2f}</td>
                    </tr>
                </table>
            </div>

            <div class="section">
                <h2>Performance Insights</h2>
                <ul>
                    <li>The open rate of <strong>{campaign['open_rate']:.2f}%</strong> indicates strong interest among the target audience.</li>
                    <li>An open to click rate of <strong>{campaign['open_to_click_rate']:.2f}%</strong> suggests effective email content and calls-to-action.</li>
                    <li>The conversion rate of <strong>{campaign['conversion_rate']:.2f}%</strong> reflects well on the campaign’s ability to drive results.</li>
                </ul>
            </div>

            <div class="section recommendations">
                <h2>Recommendations</h2>
                <ul>
                    <li>Test subject lines and sending times to increase open rates.</li>
                    <li>Segment audience further for targeted offers.</li>
                    <li>Optimize landing pages to boost conversion rates.</li>
                </ul>
            </div>
        </body>
        </html>
        """

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(html_content)

        print(f"HTML report generated and saved as '{output_path}'.")

    # -- Process
    # Load CSV generated by previous step
    df_campaigns = pd.read_csv(campaign_file_full_path, parse_dates=["campaign_date"])

    # Folder
    htmls = "data/html"

    # Generate HTML report for the first campaign

    # Generate reports
    for i in range(200, 202):
        campaign = df_campaigns.iloc[i]
        campaign_id = campaign['campaign_id']
        html_file_name = f"campaign_{campaign_id}_summary_report.html"
        html_file_full_path = f"{htmls}/{html_file_name}"
        generate_campaign_summary_html(df_campaigns, campaign_index=i, output_path=html_file_full_path)    


In [8]:
generate_campaign_summary_report_as_html()

HTML report generated and saved as 'data/html/campaign_201_summary_report.html'.
HTML report generated and saved as 'data/html/campaign_202_summary_report.html'.
